# 13 -- RAG Architecture: End-to-End Retrieval-Augmented Generation
**ServiceTitan context**: The estimating assistant RAG pipeline grounds an LLM in a contractor's price book. This notebook builds a complete RAG system with sparse retrieval, dense retrieval, re-ranking, context window management, and evaluation.

Topics: Sparse (TF-IDF) vs dense (embedding) retrieval * Re-ranking * Context management * Chunking * Hit@K / MRR evaluation

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt
import re, json, textwrap
np.random.seed(42)
print('Imports OK')

Imports OK


## 1. The Price Book Corpus (Knowledge Base)

In [2]:
# Simulate a ServiceTitan price book -- the knowledge base for RAG
PRICE_BOOK = [
    {'id':'SVC001','title':'HVAC System Tune-Up',
     'text':'Complete HVAC tune-up service including inspection of all components, '
            'filter replacement, coil cleaning, refrigerant check, and efficiency test. '
            'Recommended annually. Price: $189. Labor: 1.5 hours.'},
    {'id':'SVC002','title':'AC Refrigerant Recharge R-410A',
     'text':'Add refrigerant to air conditioning system. Includes leak check and pressure test. '
            'Price per pound: $75. Typical job 2-4 lbs. Labor: 1 hour minimum.'},
    {'id':'SVC003','title':'Furnace Heat Exchanger Inspection',
     'text':'Safety inspection of furnace heat exchanger for cracks and carbon monoxide risk. '
            'Includes combustion analysis. Price: $129. Labor: 1 hour. Critical safety service.'},
    {'id':'SVC004','title':'Tankless Water Heater Installation',
     'text':'Install tankless on-demand water heater. Includes gas line connection, '
            'venting, and commissioning. Parts not included. Labor: $650. Time: 4-6 hours.'},
    {'id':'SVC005','title':'Drain Cleaning -- Main Sewer Line',
     'text':'Clear blockage from main sewer line using electric snake or hydro-jet. '
            'Includes camera inspection. Price: $299-$499. Labor: 2-3 hours.'},
    {'id':'SVC006','title':'Water Heater Flush and Service',
     'text':'Flush sediment from tank water heater, inspect anode rod, test pressure relief valve. '
            'Extends heater life by 3-5 years. Price: $99. Labor: 45 minutes.'},
    {'id':'SVC007','title':'Electrical Panel Upgrade 200A',
     'text':'Replace outdated electrical panel with 200-amp service. Includes permit and inspection. '
            'Price: $2,200-$3,500. Labor: 8-10 hours. Permit required.'},
    {'id':'SVC008','title':'GFCI Outlet Installation',
     'text':'Install ground fault circuit interrupter outlet in kitchen or bathroom. '
            'Code-required near water. Price: $189 per outlet. Labor: 30 minutes.'},
    {'id':'SVC009','title':'Air Duct Cleaning',
     'text':'Clean all supply and return air ducts using rotary brush and HEPA vacuum. '
            'Reduces allergens, improves airflow. Price: $399 for up to 10 vents. Labor: 3 hours.'},
    {'id':'SVC010','title':'Thermostat Replacement -- Smart WiFi',
     'text':'Remove old thermostat and install Nest or Ecobee smart WiFi thermostat. '
            'Includes programming and app setup. Price: $249 labor + thermostat cost.'},
    {'id':'SVC011','title':'Garbage Disposal Replacement',
     'text':'Remove old disposal unit and install new. Includes electrical and plumbing connections. '
            'Price: $249 labor. Disposal unit extra. Labor: 1 hour.'},
    {'id':'SVC012','title':'Whole-Home Air Purifier Installation',
     'text':'Install whole-home HEPA or UV air purification system inline with HVAC. '
            'Reduces bacteria, viruses, and allergens. Price: $599-$999 installed.'},
]
print(f'Price book: {len(PRICE_BOOK)} services')
print('Sample:', PRICE_BOOK[0]['title'])

Price book: 12 services
Sample: HVAC System Tune-Up


## 2. Sparse Retrieval (TF-IDF / BM25 variant)

In [3]:
# TF-IDF: fast, no model required, great for keyword matching
# O(|V| * n_docs) index build; O(|V| * n_query) query time

docs = [f"{d['title']} {d['text']}" for d in PRICE_BOOK]
doc_ids = [d['id'] for d in PRICE_BOOK]

tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=5000)
doc_vecs = tfidf.fit_transform(docs)  # shape: (n_docs, vocab_size)
print(f'TF-IDF index: {doc_vecs.shape[0]} docs x {doc_vecs.shape[1]} terms')

def sparse_retrieve(query: str, k: int = 3) -> list:
    q_vec = tfidf.transform([query])
    scores = cosine_similarity(q_vec, doc_vecs).flatten()
    top_k = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], PRICE_BOOK[i]['title'], float(scores[i])) for i in top_k]

# Test
for q in ['AC refrigerant recharge', 'drain clogged sewer', 'smart thermostat wifi']:
    results = sparse_retrieve(q, k=2)
    print(f'Query: "{q}"')
    for rid, title, score in results:
        print(f'  {rid}: {title}  (score={score:.3f})')

TF-IDF index: 12 docs x 445 terms
Query: "AC refrigerant recharge"
  SVC002: AC Refrigerant Recharge R-410A  (score=0.374)
  SVC001: HVAC System Tune-Up  (score=0.046)
Query: "drain clogged sewer"
  SVC005: Drain Cleaning -- Main Sewer Line  (score=0.294)
  SVC012: Whole-Home Air Purifier Installation  (score=0.000)
Query: "smart thermostat wifi"
  SVC010: Thermostat Replacement -- Smart WiFi  (score=0.597)
  SVC012: Whole-Home Air Purifier Installation  (score=0.000)


## 3. Dense Retrieval (Embedding-Based -- Simulated)

In [4]:
# In production: sentence-transformers or Azure OpenAI text-embedding-3-small
# We simulate 64-dim embeddings using LSA (TruncatedSVD on TF-IDF matrix)
# This captures semantic similarity where TF-IDF fails on vocabulary mismatch

EMBED_DIM = 32
svd = TruncatedSVD(n_components=EMBED_DIM, random_state=42)
doc_embeds = svd.fit_transform(doc_vecs)          # (n_docs, EMBED_DIM)
# Normalize for cosine similarity
norms = np.linalg.norm(doc_embeds, axis=1, keepdims=True) + 1e-9
doc_embeds_norm = doc_embeds / norms

def dense_embed(text: str) -> np.ndarray:
    vec = tfidf.transform([text])
    emb = svd.transform(vec)
    return emb / (np.linalg.norm(emb) + 1e-9)

def dense_retrieve(query: str, k: int = 3) -> list:
    q_emb = dense_embed(query)                     # O(|V|) + O(d)
    scores = doc_embeds_norm @ q_emb.T             # O(n * d) dot products
    scores = scores.flatten()
    top_k = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], PRICE_BOOK[i]['title'], float(scores[i])) for i in top_k]

# Semantic query -- won't match on keywords but should match on meaning
for q in ['heating system checkup', 'water not draining', 'upgrade electrical service']:
    results = dense_retrieve(q, k=2)
    print(f'Query: "{q}"')
    for rid, title, score in results:
        print(f'  {rid}: {title}  (score={score:.3f})')

Query: "heating system checkup"
  SVC002: AC Refrigerant Recharge R-410A  (score=0.635)
  SVC012: Whole-Home Air Purifier Installation  (score=0.593)
Query: "water not draining"
  SVC004: Tankless Water Heater Installation  (score=0.940)
  SVC006: Water Heater Flush and Service  (score=0.452)
Query: "upgrade electrical service"
  SVC007: Electrical Panel Upgrade 200A  (score=0.957)
  SVC011: Garbage Disposal Replacement  (score=0.248)


## 4. Hybrid Retrieval (Sparse + Dense Fusion)

In [5]:
# RRF: Reciprocal Rank Fusion -- combines rankings without score normalization
# Score(d) = sum_r 1/(k + rank_r(d)) where k=60 is a smoothing constant
# O(n * n_retrievers): fast and parameter-free

def hybrid_retrieve(query: str, k: int = 3, rrf_k: int = 60) -> list:
    sparse_results = sparse_retrieve(query, k=len(PRICE_BOOK))
    dense_results  = dense_retrieve(query,  k=len(PRICE_BOOK))
    
    rrf_scores = defaultdict(float) if False else {}
    for rank, (rid, _, _) in enumerate(sparse_results):
        rrf_scores[rid] = rrf_scores.get(rid, 0) + 1.0 / (rrf_k + rank + 1)
    for rank, (rid, _, _) in enumerate(dense_results):
        rrf_scores[rid] = rrf_scores.get(rid, 0) + 1.0 / (rrf_k + rank + 1)
    
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    result = []
    for rid, score in ranked[:k]:
        title = next(d['title'] for d in PRICE_BOOK if d['id']==rid)
        result.append((rid, title, score))
    return result

print('Hybrid retrieval (RRF fusion):')
for q in ['my AC needs a tune-up', 'bathroom outlet near sink', 'water heater annual maintenance']:
    results = hybrid_retrieve(q, k=2)
    print(f'  Query: "{q}"')
    for rid, title, score in results:
        print(f'    {rid}: {title}')

Hybrid retrieval (RRF fusion):
  Query: "my AC needs a tune-up"
    SVC001: HVAC System Tune-Up
    SVC002: AC Refrigerant Recharge R-410A
  Query: "bathroom outlet near sink"
    SVC008: GFCI Outlet Installation
    SVC012: Whole-Home Air Purifier Installation
  Query: "water heater annual maintenance"
    SVC006: Water Heater Flush and Service
    SVC004: Tankless Water Heater Installation


## 5. Re-ranking

In [6]:
# Re-ranker: a heavier model that scores (query, doc) pairs
# Retrieve k_large=20, re-rank, keep top k_final=3
# This separates recall (retrieval) from precision (ranking)

def mock_reranker(query: str, docs: list) -> list:
    # In production: cross-encoder (BERT-style) or Azure OpenAI re-ranking API
    # We simulate by adding keyword overlap bonus on top of retrieval score
    q_words = set(query.lower().split())
    scored = []
    for rid, title, base_score in docs:
        full = next(d['text'] for d in PRICE_BOOK if d['id']==rid).lower()
        overlap = len(q_words & set(full.split())) / (len(q_words) + 1)
        rerank_score = 0.6 * base_score + 0.4 * overlap
        scored.append((rid, title, rerank_score))
    return sorted(scored, key=lambda x: x[2], reverse=True)

query = 'annual HVAC maintenance service contract'
candidates = hybrid_retrieve(query, k=6)
reranked   = mock_reranker(query, candidates)
print(f'Query: "{query}"')
print(f'\nBefore re-ranking (top 3 of {len(candidates)}):')
for r,t,s in candidates[:3]: print(f'  {r}: {t}  ({s:.4f})')
print(f'\nAfter re-ranking (top 3 of {len(candidates)}):')
for r,t,s in reranked[:3]: print(f'  {r}: {t}  ({s:.4f})')

Query: "annual HVAC maintenance service contract"

Before re-ranking (top 3 of 6):
  SVC001: HVAC System Tune-Up  (0.0328)
  SVC012: Whole-Home Air Purifier Installation  (0.0323)
  SVC007: Electrical Panel Upgrade 200A  (0.0317)

After re-ranking (top 3 of 6):
  SVC001: HVAC System Tune-Up  (0.1530)
  SVC012: Whole-Home Air Purifier Installation  (0.0194)
  SVC007: Electrical Panel Upgrade 200A  (0.0190)


## 6. Context Window Management & Prompt Assembly

In [7]:
# Context window is finite (e.g. 128K tokens for GPT-4)
# Strategy: retrieve k=10, fit as many as budget allows, truncate if needed

MAX_CONTEXT_CHARS = 2000   # ~500 tokens

def assemble_prompt(user_query: str, retrieved_docs: list,
                    max_chars: int = MAX_CONTEXT_CHARS) -> str:
    context_parts = []
    used = 0
    for rid, title, score in retrieved_docs:
        doc = next(d for d in PRICE_BOOK if d['id']==rid)
        chunk = f"[{rid}] {doc['title']}\n{doc['text']}"
        if used + len(chunk) + 10 > max_chars:
            # Truncate last doc to fit budget
            remaining = max_chars - used - 10
            if remaining > 100:
                context_parts.append(chunk[:remaining] + '...')
            break
        context_parts.append(chunk)
        used += len(chunk)
    
    context = '\n\n'.join(context_parts)
    prompt = (
        f"You are a ServiceTitan estimating assistant. "
        f"Use the price book entries below to build an estimate.\n\n"
        f"PRICE BOOK:\n{context}\n\n"
        f"CUSTOMER REQUEST: {user_query}\n\n"
        f"Draft a professional estimate with line items, prices, and total."
    )
    return prompt

query = 'Customer wants full HVAC tune-up and smart thermostat installed'
docs  = hybrid_retrieve(query, k=4)
reranked = mock_reranker(query, docs)
prompt = assemble_prompt(query, reranked[:3])
print(f'Assembled prompt ({len(prompt)} chars):')
print('-'*50)
print(prompt[:800], '...' if len(prompt)>800 else '')

Assembled prompt (870 chars):
--------------------------------------------------
You are a ServiceTitan estimating assistant. Use the price book entries below to build an estimate.

PRICE BOOK:
[SVC001] HVAC System Tune-Up
Complete HVAC tune-up service including inspection of all components, filter replacement, coil cleaning, refrigerant check, and efficiency test. Recommended annually. Price: $189. Labor: 1.5 hours.

[SVC010] Thermostat Replacement -- Smart WiFi
Remove old thermostat and install Nest or Ecobee smart WiFi thermostat. Includes programming and app setup. Price: $249 labor + thermostat cost.

[SVC012] Whole-Home Air Purifier Installation
Install whole-home HEPA or UV air purification system inline with HVAC. Reduces bacteria, viruses, and allergens. Price: $599-$999 installed.

CUSTOMER REQUEST: Customer wants full HVAC tune-up and smart thermostat instal ...


## 7. Evaluation: Hit@K and MRR

In [8]:
# Build an evaluation set: (query, relevant_doc_id)
eval_set = [
    ('AC tune-up and refrigerant check',      'SVC001'),
    ('recharge refrigerant air conditioning', 'SVC002'),
    ('clogged main sewer line',               'SVC005'),
    ('install smart thermostat',              'SVC010'),
    ('electrical panel replacement',          'SVC007'),
    ('water heater service flush',            'SVC006'),
    ('kitchen outlet GFCI',                  'SVC008'),
    ('furnace safety check',                  'SVC003'),
]

def evaluate(retriever_fn, eval_set, k_vals=(1,3,5)):
    metrics = {f'Hit@{k}':[] for k in k_vals}
    metrics['MRR'] = []
    for query, relevant in eval_set:
        results = retriever_fn(query, k=max(k_vals))
        retrieved_ids = [r[0] for r in results]
        for k in k_vals:
            hit = int(relevant in retrieved_ids[:k])
            metrics[f'Hit@{k}'].append(hit)
        rank = next((i+1 for i,r in enumerate(retrieved_ids) if r==relevant), 0)
        metrics['MRR'].append(1/rank if rank else 0)
    return {m: np.mean(v) for m, v in metrics.items()}

sparse_m  = evaluate(sparse_retrieve, eval_set)
dense_m   = evaluate(dense_retrieve,  eval_set)

def hybrid_k(q, k): return hybrid_retrieve(q, k)
hybrid_m  = evaluate(hybrid_k, eval_set)

results_df = pd.DataFrame({'Sparse(TF-IDF)': sparse_m,
                            'Dense(LSA)':     dense_m,
                            'Hybrid(RRF)':    hybrid_m}).T
print('Retrieval Evaluation:')
print(results_df.round(3).to_string())

Retrieval Evaluation:
                Hit@1  Hit@3  Hit@5  MRR
Sparse(TF-IDF)    1.0    1.0    1.0  1.0
Dense(LSA)        1.0    1.0    1.0  1.0
Hybrid(RRF)       1.0    1.0    1.0  1.0


## 8. Key RAG Design Tradeoffs

In [9]:
rows = [
    ('Sparse (TF-IDF/BM25)', 'Fast, no model needed, keyword exact match',
     'Fails on synonyms, semantic queries'),
    ('Dense (embeddings)',   'Semantic similarity, handles paraphrases',
     'Slow to build index, needs GPU for large corpora'),
    ('Hybrid (RRF)',         'Best of both, robust',
     '2x retrieval latency'),
    ('Re-ranking',           'High precision at top-k',
     'O(k) expensive model calls'),
    ('Chunk size small',     'Higher precision per chunk',
     'Splits context, loses surrounding info'),
    ('Chunk size large',     'More context per chunk',
     'Noisy retrieval, wastes token budget'),
]
print(f'  {"Method":<26} {"Pro":<44} Con')
print('-'*95)
for m,p,c_ in rows:
    print(f'  {m:<26} {p:<44} {c_}')

  Method                     Pro                                          Con
-----------------------------------------------------------------------------------------------
  Sparse (TF-IDF/BM25)       Fast, no model needed, keyword exact match   Fails on synonyms, semantic queries
  Dense (embeddings)         Semantic similarity, handles paraphrases     Slow to build index, needs GPU for large corpora
  Hybrid (RRF)               Best of both, robust                         2x retrieval latency
  Re-ranking                 High precision at top-k                      O(k) expensive model calls
  Chunk size small           Higher precision per chunk                   Splits context, loses surrounding info
  Chunk size large           More context per chunk                       Noisy retrieval, wastes token budget
